# MT-SICS Backend Hardware Validation

Run this notebook connected to a physical Mettler Toledo scale to validate
every implemented backend method. Each cell tests one method and logs the
raw MT-SICS command/response.

**Requirements:**
- Physical Mettler Toledo scale connected via USB-to-serial
- Scale powered on and warmed up (60 min)
- Weighing pan empty at start

---
## 1. Setup and Logging

In [1]:
import logging
from datetime import datetime

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO
from pylabrobot.scales import Scale
from pylabrobot.scales.mettler_toledo import MettlerToledoWXS205SDUBackend

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = f"./logs/hardware_validation/{timestamp}_validation.log"

pylabrobot.verbose(True, level=LOG_LEVEL_IO)
pylabrobot.setup_logger(log_dir=log_file, level=LOG_LEVEL_IO)

plr_logger = logging.getLogger("pylabrobot")
plr_logger.info("=== Hardware validation started ===")

2026-03-30 13:12:14,770 - pylabrobot - INFO - === Hardware validation started ===


In [2]:
# Update port for your system
backend = MettlerToledoWXS205SDUBackend(
  port="/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0"
)
scale = Scale(name="validation_scale", backend=backend, size_x=0, size_y=0, size_z=0)

await scale.setup()

2026-03-30 13:12:14,791 - pylabrobot.io.serial - INFO - Using explicitly provided port: /dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0 (for VID=1027, PID=24577)
2026-03-30 13:12:14,799 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 13:12:14,800 - pylabrobot - IO - [MT Scale] Sent command: @
2026-03-30 13:12:14,803 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 13:12:14,862 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 13:12:14,863 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 13:12:14,864 - pylabrobot - IO - [MT Scale] Sent command: I0
2026-03-30 13:12:14,865 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I0\r\n'
2026-03-30 13:12:14,893 - pylabrobot.io.serial - IO - [/dev/serial

### Verify setup populated device identity

In [3]:
print(f"Device type:    {backend.device_type}")
print(f"Serial number:  {backend.serial_number}")
print(f"Capacity:       {backend.capacity} g")
print(
  f"Supported commands ({len(backend._supported_commands)}): {sorted(backend._supported_commands)}"
)

assert backend.device_type is not None
assert backend.serial_number is not None
assert backend.capacity > 0
assert "S" in backend._supported_commands
print("PASS: setup populated all identity fields")

Device type:    WXS205SDU WXA-Bridge
Serial number:  B207696838
Capacity:       220.009 g
Supported commands (62): ['@', 'C0', 'C1', 'C2', 'C3', 'COM', 'DAT', 'FCUT', 'FSET', 'I0', 'I1', 'I10', 'I11', 'I14', 'I15', 'I16', 'I2', 'I21', 'I22', 'I23', 'I24', 'I25', 'I26', 'I3', 'I4', 'I5', 'LST', 'M01', 'M02', 'M03', 'M17', 'M18', 'M19', 'M20', 'M21', 'M27', 'M28', 'M29', 'M31', 'M32', 'M33', 'M35', 'RDB', 'S', 'SI', 'SIR', 'SIS', 'SNR', 'SR', 'T', 'TA', 'TAC', 'TI', 'TIM', 'TST0', 'TST1', 'TST2', 'TST3', 'UPD', 'USTB', 'Z', 'ZI']
PASS: setup populated all identity fields


### Supported Python methods on this device

In [4]:
methods = backend.request_supported_methods()
print(f"{len(methods)} methods available on this device:")
for m in methods:
  print(f"  {m}")

46 methods available on this device:
  cancel
  cancel_all
  clear_tare
  deserialize
  get_all_instances
  get_weight
  measure_temperature
  read_dynamic_weight
  read_stable_weight
  read_stable_weight_repeat_on_change
  read_weight
  read_weight_value_immediately
  request_adjustment_history
  request_auto_zero
  request_capacity
  request_date
  request_device_id
  request_device_info
  request_device_type
  request_environment_condition
  request_model_designation
  request_net_weight_with_status
  request_serial_number
  request_software_material_number
  request_software_version
  request_supported_methods
  request_tare_weight
  request_time
  request_update_rate
  request_uptime
  request_weighing_mode
  send_command
  serialize
  set_display_text
  set_host_unit_grams
  set_weight_display
  setup
  stop
  tare
  tare_immediately
  tare_stable
  tare_timeout
  zero
  zero_immediately
  zero_stable
  zero_timeout


### I0 - Discover all implemented commands

This tells us exactly which MT-SICS commands this specific device supports,
resolving whether SC, C, D are truly unsupported or just miscategorized by level.

In [5]:
from collections import defaultdict

# I0 is a multi-response command (B status for each command, A for the last)
responses = await backend.send_command("I0")

print(f"Total commands implemented: {len(responses)}")
print()

# Group by level
by_level = defaultdict(list)
for resp in responses:
  # Format: I0 B/A <level> <"command">
  if len(resp.data) >= 2:
    level = resp.data[0]
    cmd_name = resp.data[1]
    by_level[level].append(cmd_name)

for level in sorted(by_level.keys()):
  cmds = sorted(by_level[level])
  print(f"Level {level} ({len(cmds)} commands): {', '.join(cmds)}")

# Check which commands we use that might not be supported
our_commands = [
  "@",
  "C",
  "D",
  "DW",
  "I0",
  "I1",
  "I2",
  "I4",
  "I50",
  "M21",
  "M28",
  "S",
  "SC",
  "SI",
  "T",
  "TA",
  "TAC",
  "TI",
  "Z",
  "ZC",
  "ZI",
  "TC",
]
all_device_cmds = set()
for cmds in by_level.values():
  all_device_cmds.update(cmds)

print()
print("Commands we use vs device support:")
for cmd in sorted(our_commands):
  status = "supported" if cmd in all_device_cmds else "NOT SUPPORTED"
  print(f"  {cmd}: {status}")

2026-03-30 13:12:16,056 - pylabrobot - IO - [MT Scale] Sent command: I0
2026-03-30 13:12:16,058 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I0\r\n'
2026-03-30 13:12:16,093 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I0"\r\n'
2026-03-30 13:12:16,094 - pylabrobot - IO - [MT Scale] Received response: b'I0 B 0 "I0"\r\n'
2026-03-30 13:12:16,109 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I1"\r\n'
2026-03-30 13:12:16,109 - pylabrobot - IO - [MT Scale] Received response: b'I0 B 0 "I1"\r\n'
2026-03-30 13:12:16,125 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I2"\r\n'
2026-03-30 13:12:16,126 - pylabrobot - IO - [MT Scale] Received response: b'I0 B 0 "I2"\r\n'
2026-03-30 13:12:16,127 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0]

Total commands implemented: 62

Level 0 (12 commands): @, I0, I1, I2, I3, I4, I5, S, SI, SIR, Z, ZI
Level 1 (5 commands): SR, T, TA, TAC, TI
Level 2 (40 commands): C0, C1, C2, C3, COM, DAT, I10, I11, I14, I15, I16, I21, I22, I23, I24, I25, I26, M01, M02, M03, M17, M18, M19, M20, M21, M27, M28, M29, M31, M32, M33, M35, SIS, SNR, TIM, TST0, TST1, TST2, TST3, UPD
Level 3 (5 commands): FCUT, FSET, LST, RDB, USTB

Commands we use vs device support:
  @: supported
  C: NOT SUPPORTED
  D: NOT SUPPORTED
  DW: NOT SUPPORTED
  I0: supported
  I1: supported
  I2: supported
  I4: supported
  I50: NOT SUPPORTED
  M21: supported
  M28: supported
  S: supported
  SC: NOT SUPPORTED
  SI: supported
  T: supported
  TA: supported
  TAC: supported
  TC: NOT SUPPORTED
  TI: supported
  Z: supported
  ZC: NOT SUPPORTED
  ZI: supported


### Device identity and diagnostics (Batch 1)

These commands complete the device identity picture beyond I2/I4.

In [6]:
# I3 - Software version (Level 0, always available)
sw_version = await backend.request_software_version()
print(f"Software version: {sw_version}")

# I5 - Software material number (Level 0, always available)
sw_material = await backend.request_software_material_number()
print(f"Software material number: {sw_material}")

# I10 - Device ID (user-assignable name)
if "I10" in backend._supported_commands:
  device_id = await backend.request_device_id()
  print(f"Device ID: '{device_id}'")
else:
  print("SKIP: I10 not supported")

# I11 - Model designation
if "I11" in backend._supported_commands:
  model = await backend.request_model_designation()
  print(f"Model designation: {model}")
else:
  print("SKIP: I11 not supported")

# I15 - Uptime
if "I15" in backend._supported_commands:
  uptime = await backend.request_uptime()
  print(f"Uptime: {uptime['days']}d {uptime['hours']}h {uptime['minutes']}m {uptime['seconds']}s")
else:
  print("SKIP: I15 not supported")

# DAT - Date
if "DAT" in backend._supported_commands:
  date = await backend.request_date()
  print(f"Device date: {date}")
else:
  print("SKIP: DAT not supported")

# TIM - Time
if "TIM" in backend._supported_commands:
  time_str = await backend.request_time()
  print(f"Device time: {time_str}")
else:
  print("SKIP: TIM not supported")

# I14 - Comprehensive device info (query each category separately)
if "I14" in backend._supported_commands:
  category_names = {
    0: "Instrument configuration",
    1: "Instrument descriptions",
    2: "SW identification numbers",
    3: "SW versions",
    4: "Serial numbers",
    5: "TDNR numbers",
  }
  print("\nDevice info (I14):")
  for cat, name in category_names.items():
    try:
      info = await backend.request_device_info(category=cat)
      print(f"  Category {cat} ({name}):")
      for resp in info:
        print(f"    {resp.command} {resp.status} {' '.join(resp.data)}")
    except Exception as e:
      print(f"  Category {cat} ({name}): {e}")
else:
  print("SKIP: I14 not supported")

print("\nPASS: Batch 1 device identity commands completed")

2026-03-30 13:12:16,984 - pylabrobot - IO - [MT Scale] Sent command: I3
2026-03-30 13:12:16,986 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I3\r\n'
2026-03-30 13:12:17,036 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I3 A "1.10 18.6.4.1361.772"\r\n'
2026-03-30 13:12:17,038 - pylabrobot - IO - [MT Scale] Received response: b'I3 A "1.10 18.6.4.1361.772"\r\n'
2026-03-30 13:12:17,038 - pylabrobot - IO - [MT Scale] Sent command: I5
2026-03-30 13:12:17,041 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I5\r\n'
2026-03-30 13:12:17,084 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I5 A "11671158C"\r\n'
2026-03-30 13:12:17,084 - pylabrobot - IO - [MT Scale] Received response: b'I5 A "11671158C"\r\n'
2026-03-30 13:12:17,085 - pylabrobot - IO - [MT Scale] Sent command: I10
2026-03-30 13:12:17,087 - p

Software version: 1.10 18.6.4.1361.772
Software material number: 11671158C
Device ID: ''
Model designation: WXS205SDU


2026-03-30 13:12:17,196 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I15 A 224\r\n'
2026-03-30 13:12:17,197 - pylabrobot - IO - [MT Scale] Received response: b'I15 A 224\r\n'
2026-03-30 13:12:17,201 - pylabrobot - IO - [MT Scale] Sent command: DAT
2026-03-30 13:12:17,202 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'DAT\r\n'


Uptime: 224d 0h 0m 0s


2026-03-30 13:12:17,244 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'DAT A 04 01 2000\r\n'
2026-03-30 13:12:17,245 - pylabrobot - IO - [MT Scale] Received response: b'DAT A 04 01 2000\r\n'
2026-03-30 13:12:17,247 - pylabrobot - IO - [MT Scale] Sent command: TIM
2026-03-30 13:12:17,248 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TIM\r\n'
2026-03-30 13:12:17,276 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TIM A 20 17 43\r\n'
2026-03-30 13:12:17,277 - pylabrobot - IO - [MT Scale] Received response: b'TIM A 20 17 43\r\n'
2026-03-30 13:12:17,278 - pylabrobot - IO - [MT Scale] Sent command: I14 0
2026-03-30 13:12:17,279 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I14 0\r\n'
2026-03-30 13:12:17,324 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline

Device date: 04
Device time: 20

Device info (I14):
  Category 0 (Instrument configuration):
    I14 A 0 1 Bridge
  Category 1 (Instrument descriptions):
    I14 A 1 1 WXS205SDU


2026-03-30 13:12:17,420 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I14 A 2 1 "11671158C"\r\n'
2026-03-30 13:12:17,421 - pylabrobot - IO - [MT Scale] Received response: b'I14 A 2 1 "11671158C"\r\n'
2026-03-30 13:12:17,421 - pylabrobot - IO - [MT Scale] Sent command: I14 3
2026-03-30 13:12:17,422 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I14 3\r\n'


  Category 2 (SW identification numbers):
    I14 A 2 1 11671158C


2026-03-30 13:12:17,468 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I14 A 3 1 "1.10"\r\n'
2026-03-30 13:12:17,468 - pylabrobot - IO - [MT Scale] Received response: b'I14 A 3 1 "1.10"\r\n'
2026-03-30 13:12:17,469 - pylabrobot - IO - [MT Scale] Sent command: I14 4
2026-03-30 13:12:17,470 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I14 4\r\n'
2026-03-30 13:12:17,516 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I14 A 4 1 "B207696838"\r\n'
2026-03-30 13:12:17,517 - pylabrobot - IO - [MT Scale] Received response: b'I14 A 4 1 "B207696838"\r\n'
2026-03-30 13:12:17,517 - pylabrobot - IO - [MT Scale] Sent command: I14 5
2026-03-30 13:12:17,518 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I14 5\r\n'
2026-03-30 13:12:17,564 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-

  Category 3 (SW versions):
    I14 A 3 1 1.10
  Category 4 (Serial numbers):
    I14 A 4 1 B207696838
  Category 5 (TDNR numbers):
    I14 A 5 1 18.6.4.1361.772

PASS: Batch 1 device identity commands completed


---
## 2. Core Protocol Validation

These Level 0/1 commands have been validated across multiple hardware runs.
This cell consolidates them into a single pass/fail check.

In [ ]:
# @ - Cancel (returns serial number)
sn = await backend.cancel()
assert sn == backend.serial_number, f"cancel() returned {sn}, expected {backend.serial_number}"

# I4 - Serial number
sn2 = await backend.request_serial_number()
assert sn2 == sn

# I2 - Device type and capacity
dt = await backend.request_device_type()
assert len(dt) > 0
cap = await backend.request_capacity()
assert cap > 0

# S - Stable weight
w_stable = await backend.read_stable_weight()
assert isinstance(w_stable, float)

# SI - Weight immediately
w_imm = await backend.read_weight_value_immediately()
assert isinstance(w_imm, float)

# Z - Zero (stable) then read
await scale.zero(timeout="stable")
w_after_zero = await backend.read_weight_value_immediately()
assert abs(w_after_zero) < 0.001, f"Expected ~0 after zero, got {w_after_zero}"

# ZI - Zero immediately then read
await scale.zero(timeout=0)
w_after_zi = await backend.read_weight_value_immediately()
assert abs(w_after_zi) < 0.01

# TA - Request tare weight
tare = await scale.request_tare_weight()
assert isinstance(tare, float)

# TAC - Clear tare
await backend.clear_tare()
tare_after = await scale.request_tare_weight()
assert abs(tare_after) < 0.001

print("PASS: all core Level 0/1 commands validated")

---
## 3. Level 1 Commands (Elementary)

**Place a known weight on the scale for tare tests.**

### T - Tare (stable)

In [15]:
input("Place a container on the scale and press Enter...")
await scale.zero(timeout="stable")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Tare weight: {tare} g")
assert tare > 0, f"Expected positive tare, got {tare}"
print("PASS: tare stores container weight")

Place a container on the scale and press Enter... 


2026-03-30 13:12:23,026 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 13:12:23,028 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 13:12:23,414 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 13:12:23,415 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 13:12:23,416 - pylabrobot - IO - [MT Scale] Sent command: T
2026-03-30 13:12:23,417 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 13:12:23,830 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 13:12:23,831 - pylabrobot - IO - [MT Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 13:12:23,831 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 13:12:23,833 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTD

Tare weight: 0.0 g


AssertionError: Expected positive tare, got 0.0

### TA - Request tare weight

In [16]:
tare = await scale.request_tare_weight()
print(f"Tare weight from memory: {tare} g")
assert isinstance(tare, float)
print("PASS: tare weight readable from memory")

2026-03-30 13:12:26,003 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 13:12:26,006 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 13:12:26,084 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 13:12:26,086 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Tare weight from memory: 0.0 g
PASS: tare weight readable from memory


### TAC - Clear tare

In [17]:
await backend.clear_tare()
tare_after_clear = await scale.request_tare_weight()
print(f"Tare after clear: {tare_after_clear} g")
assert abs(tare_after_clear) < 0.001, f"Expected ~0 after clear, got {tare_after_clear}"
print("PASS: clear tare resets tare to 0")

2026-03-30 13:12:26,799 - pylabrobot - IO - [MT Scale] Sent command: TAC
2026-03-30 13:12:26,801 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TAC\r\n'
2026-03-30 13:12:26,836 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TAC A\r\n'
2026-03-30 13:12:26,837 - pylabrobot - IO - [MT Scale] Received response: b'TAC A\r\n'
2026-03-30 13:12:26,839 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 13:12:26,841 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 13:12:26,964 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 13:12:26,965 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Tare after clear: 0.0 g
PASS: clear tare resets tare to 0


---
## 4. Level 2 Commands (Extended)

### M21 - Set host unit to grams

In [22]:
if "M21" in backend._supported_commands:
  await backend.set_host_unit_grams()
  print("PASS: host unit set to grams")
else:
  print("SKIP: M21 not supported on this device")

2026-03-30 13:12:34,916 - pylabrobot - IO - [MT Scale] Sent command: M21 0 0
2026-03-30 13:12:34,918 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M21 0 0\r\n'
2026-03-30 13:12:34,957 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M21 A\r\n'
2026-03-30 13:12:34,958 - pylabrobot - IO - [MT Scale] Received response: b'M21 A\r\n'


PASS: host unit set to grams


### I50 - Remaining weighing range (multi-response)

In [23]:
if "I50" in backend._supported_commands:
  remaining = await backend.request_remaining_weighing_range()
  print(f"Remaining weighing range: {remaining} g")
  assert remaining > 0
  assert remaining <= backend.capacity
  print("PASS: remaining range is positive and within capacity")
else:
  print("SKIP: I50 not supported on this device")

SKIP: I50 not supported on this device


### M28 - Temperature sensor

In [24]:
if "M28" in backend._supported_commands:
  temp = await backend.measure_temperature()
  print(f"Scale temperature: {temp} C")
  assert 5 < temp < 50, f"Temperature {temp} C outside reasonable range"
  print("PASS: measure_temperature works - I0 correctly predicted M28 support")
else:
  print("SKIP: M28 not supported on this device")

2026-03-30 13:12:38,030 - pylabrobot - IO - [MT Scale] Sent command: M28
2026-03-30 13:12:38,034 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M28\r\n'
2026-03-30 13:12:38,072 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M28 B 1 20.1\r\n'
2026-03-30 13:12:38,073 - pylabrobot - IO - [MT Scale] Received response: b'M28 B 1 20.1\r\n'
2026-03-30 13:12:38,089 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M28 A 2 19.6\r\n'
2026-03-30 13:12:38,094 - pylabrobot - IO - [MT Scale] Received response: b'M28 A 2 19.6\r\n'


Scale temperature: 20.1 C
PASS: measure_temperature works - I0 correctly predicted M28 support


### Batch 2 - Configuration queries (read-only)

These commands read device configuration without modifying anything.

In [25]:
# M01 - Weighing mode (read-only query)
if "M01" in backend._supported_commands:
  mode = await backend.request_weighing_mode()
  mode_names = {0: "Normal/Universal", 1: "Dosing", 2: "Sensor", 3: "Check weighing", 6: "Raw/No filter"}
  print(f"Weighing mode: {mode} ({mode_names.get(mode, 'unknown')})")
else:
  print("SKIP: M01 not supported")

# M02 - Environment condition (read-only query)
if "M02" in backend._supported_commands:
  env = await backend.request_environment_condition()
  env_names = {0: "Very stable", 1: "Stable", 2: "Standard", 3: "Unstable", 4: "Very unstable", 5: "Automatic"}
  print(f"Environment condition: {env} ({env_names.get(env, 'unknown')})")
else:
  print("SKIP: M02 not supported")

# M03 - Auto zero (read-only query)
if "M03" in backend._supported_commands:
  auto_zero = await backend.request_auto_zero()
  print(f"Auto zero: {auto_zero} ({'on' if auto_zero else 'off'})")
else:
  print("SKIP: M03 not supported")

# M27 - Adjustment history (read-only query)
if "M27" in backend._supported_commands:
  history = await backend.request_adjustment_history()
  print(f"\nAdjustment history ({len(history)} entries):")
  for resp in history:
    print(f"  {resp.command} {resp.status} {' '.join(resp.data)}")
else:
  print("SKIP: M27 not supported")

# UPD - Update rate (read-only query)
if "UPD" in backend._supported_commands:
  rate = await backend.request_update_rate()
  print(f"\nUpdate rate: {rate} values/s")
else:
  print("SKIP: UPD not supported")

# SIS - Net weight with status (read-only query)
if "SIS" in backend._supported_commands:
  sis_resp = await backend.request_net_weight_with_status()
  print(f"\nNet weight with status: {sis_resp.command} {sis_resp.status} {' '.join(sis_resp.data)}")
else:
  print("SKIP: SIS not supported")

# LST - Current user settings (read-only, lists all configurable settings)
if "LST" in backend._supported_commands:
  try:
    lst_responses = await backend.send_command("LST")
    print(f"\nCurrent user settings ({len(lst_responses)} lines):")
    for resp in lst_responses[:10]:
      print(f"  {resp.command} {resp.status} {' '.join(resp.data)}")
    if len(lst_responses) > 10:
      print(f"  ... ({len(lst_responses)} lines total)")
  except Exception as e:
    print(f"LST: {e}")
else:
  print("SKIP: LST not supported")

# RDB - Readability (read-only)
if "RDB" in backend._supported_commands:
  try:
    rdb_responses = await backend.send_command("RDB")
    print(f"\nReadability: {rdb_responses[0].command} {rdb_responses[0].status} {' '.join(rdb_responses[0].data)}")
  except Exception as e:
    print(f"RDB: {e}")
else:
  print("SKIP: RDB not supported")

# USTB - User defined stability criteria (read-only query)
if "USTB" in backend._supported_commands:
  try:
    ustb_responses = await backend.send_command("USTB")
    print(f"Stability criteria: {ustb_responses[0].command} {ustb_responses[0].status} {' '.join(ustb_responses[0].data)}")
  except Exception as e:
    print(f"USTB: {e}")
else:
  print("SKIP: USTB not supported")

# FCUT - Filter cut-off frequency (read-only query)
if "FCUT" in backend._supported_commands:
  try:
    fcut_responses = await backend.send_command("FCUT")
    print(f"Filter cut-off: {fcut_responses[0].command} {fcut_responses[0].status} {' '.join(fcut_responses[0].data)}")
  except Exception as e:
    print(f"FCUT: {e}")
else:
  print("SKIP: FCUT not supported")

print("\nPASS: Batch 2+3 read-only configuration queries completed")

2026-03-30 13:12:42,172 - pylabrobot - IO - [MT Scale] Sent command: M01
2026-03-30 13:12:42,174 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M01\r\n'
2026-03-30 13:12:42,198 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M01 A 0\r\n'
2026-03-30 13:12:42,199 - pylabrobot - IO - [MT Scale] Received response: b'M01 A 0\r\n'
2026-03-30 13:12:42,201 - pylabrobot - IO - [MT Scale] Sent command: M02
2026-03-30 13:12:42,202 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M02\r\n'
2026-03-30 13:12:42,230 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M02 A 2\r\n'
2026-03-30 13:12:42,230 - pylabrobot - IO - [MT Scale] Received response: b'M02 A 2\r\n'
2026-03-30 13:12:42,231 - pylabrobot - IO - [MT Scale] Sent command: M03
2026-03-30 13:12:42,232 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI

Weighing mode: 0 (Normal/Universal)
Environment condition: 2 (Standard)
Auto zero: 0 (off)


2026-03-30 13:12:43,061 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 1 13 05 2020 17 17 0 ""\r\n'
2026-03-30 13:12:43,062 - pylabrobot - IO - [MT Scale] Received response: b'M27 B 1 13 05 2020 17 17 0 ""\r\n'
2026-03-30 13:12:43,094 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 2 02 12 2019 20 02 0 ""\r\n'
2026-03-30 13:12:43,096 - pylabrobot - IO - [MT Scale] Received response: b'M27 B 2 02 12 2019 20 02 0 ""\r\n'
2026-03-30 13:12:43,128 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 3 11 10 2019 17 05 0 ""\r\n'
2026-03-30 13:12:43,130 - pylabrobot - IO - [MT Scale] Received response: b'M27 B 3 11 10 2019 17 05 0 ""\r\n'
2026-03-30 13:12:43,158 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 4 11 10 2019 17 01 0 ""\r\n'
2026-03-30 13:12:43,159 - pylabrobot - IO -


Adjustment history (50 entries):
  M27 B 1 13 05 2020 17 17 0 
  M27 B 2 02 12 2019 20 02 0 
  M27 B 3 11 10 2019 17 05 0 
  M27 B 4 11 10 2019 17 01 0 
  M27 B 5 14 03 2019 20 22 0 
  M27 B 6 14 03 2019 20 18 0 
  M27 B 7 17 10 2018 17 38 0 
  M27 B 8 17 10 2018 17 37 0 
  M27 B 9 17 10 2018 17 35 0 
  M27 B 10 20 09 2018 21 23 0 
  M27 B 11 15 03 2018 23 51 0 
  M27 B 12 16 12 2017 00 20 0 
  M27 B 13 16 12 2017 00 18 0 
  M27 B 14 18 10 2017 12 06 0 
  M27 B 15 18 10 2017 11 58 0 
  M27 B 16 05 09 2017 22 40 0 
  M27 B 17 21 06 2017 18 39 0 
  M27 B 18 30 03 2017 19 24 0 
  M27 B 19 09 03 2017 23 21 0 
  M27 B 20 09 03 2017 23 19 0 
  M27 B 21 20 12 2016 21 55 0 
  M27 B 22 19 10 2016 17 36 0 
  M27 B 23 19 10 2016 17 33 0 
  M27 B 24 18 10 2016 21 00 0 
  M27 B 25 18 10 2016 18 47 0 
  M27 B 26 18 10 2016 15 33 0 
  M27 B 27 18 10 2016 14 32 0 
  M27 B 28 18 10 2016 14 28 0 
  M27 B 29 18 10 2016 13 41 0 
  M27 B 30 21 09 2016 15 15 0 
  M27 B 31 04 03 2016 22 28 0 
  M27 B 32 01 

2026-03-30 13:12:44,900 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M01 0\r\n'
2026-03-30 13:12:44,902 - pylabrobot - IO - [MT Scale] Received response: b'LST B M01 0\r\n'
2026-03-30 13:12:44,918 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M02 2\r\n'
2026-03-30 13:12:44,921 - pylabrobot - IO - [MT Scale] Received response: b'LST B M02 2\r\n'
2026-03-30 13:12:44,932 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M03 0\r\n'
2026-03-30 13:12:44,933 - pylabrobot - IO - [MT Scale] Received response: b'LST B M03 0\r\n'
2026-03-30 13:12:44,948 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M17 00 00 00 0\r\n'
2026-03-30 13:12:44,949 - pylabrobot - IO - [MT Scale] Received response: b'LST B M17 00 00 00 0\r\n'
2026-03-30 13:12:44,964 - pylabrobot.io.serial - IO - [/de


Current user settings (24 lines):
  LST B C0 0 0 
  LST B FCUT 0.000
  LST B I10 
  LST B M01 0
  LST B M02 2
  LST B M03 0
  LST B M17 00 00 00 0
  LST B M18 1
  LST B M19 10.00000 g
  LST B M20 200.00000 g
  ... (24 lines total)

Readability: RDB A 5
Stability criteria: USTB B 0 3.600 1.100


2026-03-30 13:12:45,493 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'FCUT A 0.000\r\n'
2026-03-30 13:12:45,494 - pylabrobot - IO - [MT Scale] Received response: b'FCUT A 0.000\r\n'


Filter cut-off: FCUT A 0.000

PASS: Batch 2+3 read-only configuration queries completed


---
## 5. Frontend Integration

Test the Scale frontend methods that delegate to the backend.

In [26]:
# Zero via frontend
await scale.zero(timeout="stable")
w = await scale.read_weight(timeout=0)
print(f"Frontend zero + read: {w} g")
assert abs(w) < 0.001

# Tare via frontend
input("Place a container on the scale and press Enter...")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Frontend tare weight: {tare} g")
assert tare > 0

# Read via frontend
w = await scale.read_weight(timeout="stable")
print(f"Frontend read (should be ~0 with container tared): {w} g")
assert abs(w) < 0.1

print("PASS: all frontend methods delegate correctly")

2026-03-30 13:12:45,621 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 13:12:45,625 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 13:12:46,019 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 13:12:46,020 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 13:12:46,022 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 13:12:46,024 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 13:12:46,067 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 13:12:46,069 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Frontend zero + read: 0.0 g


Place a container on the scale and press Enter... 


2026-03-30 13:12:47,098 - pylabrobot - IO - [MT Scale] Sent command: T
2026-03-30 13:12:47,100 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 13:12:47,409 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 13:12:47,410 - pylabrobot - IO - [MT Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 13:12:47,411 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 13:12:47,412 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 13:12:47,505 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 13:12:47,506 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Frontend tare weight: 0.0 g


AssertionError: 

---
## 6. Performance

In [27]:
import time

import numpy as np

# Stable read latency
times_stable = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout="stable")
  t1 = time.monotonic_ns()
  times_stable.append((t1 - t0) / 1e6)

# Immediate read latency
times_immediate = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout=0)
  t1 = time.monotonic_ns()
  times_immediate.append((t1 - t0) / 1e6)

print(f"Stable read:    {np.mean(times_stable):.2f} +/- {np.std(times_stable):.2f} ms")
print(f"Immediate read: {np.mean(times_immediate):.2f} +/- {np.std(times_immediate):.2f} ms")

2026-03-30 13:12:49,989 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 13:12:49,991 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 13:12:50,064 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 13:12:50,065 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 13:12:50,065 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 13:12:50,067 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 13:12:50,159 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 13:12:50,159 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 13:12:50,160 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 13:12:50,161 - pylabrobot.io.serial - IO - [

Stable read:    95.74 +/- 9.23 ms
Immediate read: 42.95 +/- 7.21 ms


---
## 7. Teardown

In [28]:
plr_logger.info("=== Hardware validation ended ===")
await scale.stop()
print(f"Log file: {log_file}")

2026-03-30 13:12:51,402 - pylabrobot - INFO - === Hardware validation ended ===
2026-03-30 13:12:51,413 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 13:12:51,420 - pylabrobot - IO - [MT Scale] Sent command: @
2026-03-30 13:12:51,423 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 13:12:51,488 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 13:12:51,491 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 13:12:51,493 - pylabrobot - INFO - [MT Scale] Disconnected from /dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0


Log file: ./logs/hardware_validation/2026-03-30_13-12-14_validation.log
